# Setup

In [1]:
pip install mlflow dagshub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub

dagshub.init(repo_owner='adzid23', repo_name='Freeuni_ML_Fraud_Detection', mlflow=True)

Accessing as adzid23

Initialized MLflow to track repo "adzid23/Freeuni_ML_Fraud_Detection"

Repository adzid23/Freeuni_ML_Fraud_Detection initialized!

# Load Test Data

In [3]:
DATA_PATH = '/kaggle/input/competitions/ieee-fraud-detection'

test_transaction = pd.read_csv(f'{DATA_PATH}/test_transaction.csv')
test_identity    = pd.read_csv(f'{DATA_PATH}/test_identity.csv')

test = test_transaction.merge(test_identity, on='TransactionID', how='left')
test_ids = test['TransactionID'].copy()

print(f"Test shape: {test.shape}")
print(f"Test IDs: {len(test_ids)}")

Test shape: (506691, 433)
Test IDs: 506691


# Load Model from Registry

In [4]:
model_name    = "XGBoost_Fraud_Pipeline_Run2"
model_version = "2"

model_uri  = f"models:/{model_name}/{model_version}"
pipeline   = mlflow.sklearn.load_model(model_uri)

print(f"Loaded model: {model_name} v{model_version}")
print(f"Pipeline steps: {[step[0] for step in pipeline.steps]}")

Loaded model: XGBoost_Fraud_Pipeline_Run2 v2
Pipeline steps: ['imputer', 'model']


# Generate Predictions

In [5]:
train_transaction = pd.read_csv(f'{DATA_PATH}/train_transaction.csv')
train_identity    = pd.read_csv(f'{DATA_PATH}/train_identity.csv')
train_df = train_transaction.merge(train_identity, on='TransactionID', how='left')

train_X = train_df.drop(columns=['isFraud', 'TransactionID'])

null_frac    = train_X.isnull().mean()
cols_to_drop = null_frac[null_frac > 0.5].index.tolist()

train_X = train_X.drop(columns=cols_to_drop + ['TransactionDT'], errors='ignore')

cat_cols = train_X.select_dtypes(include='object').columns.tolist()
cat_freq_maps = {}
for col in cat_cols:
    cat_freq_maps[col] = train_X[col].value_counts(normalize=True).to_dict()

card1_stats = train_X.groupby('card1')['TransactionAmt'].agg(['mean','std']).reset_index()
card1_stats.columns = ['card1','card1_amt_mean','card1_amt_std']

card2_stats = train_X.groupby('card2')['TransactionAmt'].agg(['mean','std']).reset_index()
card2_stats.columns = ['card2','card2_amt_mean','card2_amt_std']

train_X['product_card4'] = (train_df['ProductCD'].astype(str) + '_' +
                             train_df['card4'].astype(str))
prod_freq_map = train_X['product_card4'].value_counts(normalize=True).to_dict()

print("Preprocessing maps computed from training data")
print(f"  Cols to drop: {len(cols_to_drop)}")
print(f"  Cat cols encoded: {len(cat_freq_maps)}")

Preprocessing maps computed from training data
  Cols to drop: 214
  Cat cols encoded: 9


# Generate Predictions

In [6]:
X_test = test.drop(columns=['TransactionID'], errors='ignore')

X_test = X_test.drop(columns=cols_to_drop + ['TransactionDT'], errors='ignore')

X_test['hour']               = (test['TransactionDT'] // 3600) % 24
X_test['day']                = (test['TransactionDT'] // (3600 * 24)) % 7
X_test['is_night']           = ((X_test['hour'] >= 0) & (X_test['hour'] <= 6)).astype(int)
X_test['is_weekend']         = (X_test['day'] >= 5).astype(int)

X_test['TransactionAmt_log']     = np.log1p(test['TransactionAmt'])
X_test['TransactionAmt_cents']   = (test['TransactionAmt'] * 100).astype(int) % 100
X_test['TransactionAmt_isround'] = (test['TransactionAmt'] % 1 == 0).astype(int)

X_test['is_gmail']   = (test['P_emaildomain'] == 'gmail.com').astype(int)
X_test['is_yahoo']   = test['P_emaildomain'].str.contains('yahoo', na=False).astype(int)
X_test['same_email'] = (test['P_emaildomain'] == test['R_emaildomain']).astype(int)

for col in cat_cols:
    if col in X_test.columns:
        X_test[col] = X_test[col].map(cat_freq_maps[col]).fillna(0)

X_test = X_test.merge(card1_stats, on='card1', how='left')
X_test = X_test.merge(card2_stats, on='card2', how='left')
X_test['card1_amt_diff']  = X_test['TransactionAmt'] - X_test['card1_amt_mean']
X_test['card1_amt_ratio'] = X_test['TransactionAmt'] / (X_test['card1_amt_mean'] + 1e-8)

X_test['product_card4_freq'] = (test['ProductCD'].astype(str) + '_' +
                                 test['card4'].astype(str))
X_test['product_card4_freq'] = X_test['product_card4_freq'].map(prod_freq_map).fillna(0)

feature_names = pipeline.named_steps['imputer'].feature_names_in_
X_test = X_test.reindex(columns=feature_names, fill_value=0)

print(f"Test features shape: {X_test.shape}")
print(f"Expected features:   {len(feature_names)}")

Test features shape: (506691, 150)
Expected features:   150


# Save Submission

In [7]:
predictions = pipeline.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud':       predictions
})

submission.to_csv('submission.csv', index=False)
print(f"Submission shape: {submission.shape}")
print(f"Mean prediction: {predictions.mean():.4f}")
print(submission.head(10))
print("submission.csv saved!")

Submission shape: (506691, 2)
Mean prediction: 0.1293
   TransactionID   isFraud
0        3663549  0.009815
1        3663550  0.007568
2        3663551  0.013672
3        3663552  0.014684
4        3663553  0.022309
5        3663554  0.055706
6        3663555  0.016813
7        3663556  0.220060
8        3663557  0.000622
9        3663558  0.182659
submission.csv saved!
